In [69]:
import pandas as pd
import numpy as np

# Load datasets
sales = pd.read_csv("bm_sales.csv")
customers = pd.read_csv("bm_customers.csv")
inventory = pd.read_csv("bm_inventory.csv")
promotions = pd.read_csv("bm_promotions.csv")
skus = pd.read_csv("bm_skus.csv")
stores = pd.read_csv("bm_stores.csv")

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [70]:
# Dataset overview

datasets = {
    "Sales": sales,
    "Customers": customers,
    "Inventory": inventory,
    "Promotions": promotions,
    "SKUs": skus,
    "Stores": stores
}

print("=" * 60)
print("ENTERPRISE RETAIL DATASET OVERVIEW")
print("=" * 60)

for name, df in datasets.items():
    print(f"{name:<12} | Rows: {df.shape[0]:>10,} | Columns: {df.shape[1]:>3}")

print("=" * 60)

ENTERPRISE RETAIL DATASET OVERVIEW
Sales        | Rows:    641,843 | Columns:   9
Customers    | Rows:      5,000 | Columns:   7
Inventory    | Rows:      8,735 | Columns:   7
Promotions   | Rows:         33 | Columns:   6
SKUs         | Rows:        200 | Columns:   7
Stores       | Rows:         50 | Columns:   5


In [71]:
# Data Quality Audit
# Check missing values, duplicate rows and data types

print("=" * 70)
print("DATA QUALITY AUDIT")
print("=" * 70)

for name, df in datasets.items():

    print(f"\n{name.upper()}")
    print("-" * 70)

    print(f"Rows              : {len(df):,}")
    print(f"Duplicate rows    : {df.duplicated().sum():,}")
    print(f"Missing values    : {df.isnull().sum().sum():,}")

    missing = df.isnull().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("\nMissing values by column:")
        print(missing)
    else:
        print("\nNo missing values detected.")

DATA QUALITY AUDIT

SALES
----------------------------------------------------------------------
Rows              : 641,843
Duplicate rows    : 45
Missing values    : 159,821

Missing values by column:
customer_id    159821
dtype: int64

CUSTOMERS
----------------------------------------------------------------------
Rows              : 5,000
Duplicate rows    : 0
Missing values    : 0

No missing values detected.

INVENTORY
----------------------------------------------------------------------
Rows              : 8,735
Duplicate rows    : 0
Missing values    : 0

No missing values detected.

PROMOTIONS
----------------------------------------------------------------------
Rows              : 33
Duplicate rows    : 0
Missing values    : 0

No missing values detected.

SKUS
----------------------------------------------------------------------
Rows              : 200
Duplicate rows    : 0
Missing values    : 0

No missing values detected.

STORES
---------------------------------------

In [72]:
# Root Cause Analysis: Missing Customer IDs

total_sales = len(sales)
missing_customer = sales["customer_id"].isna().sum()
known_customer = sales["customer_id"].notna().sum()

missing_pct = (missing_customer / total_sales) * 100

print("=" * 60)
print("CUSTOMER ID ROOT CAUSE ANALYSIS")
print("=" * 60)

print(f"Total sales records       : {total_sales:,}")
print(f"Known customer records    : {known_customer:,}")
print(f"Anonymous customer records: {missing_customer:,}")
print(f"Anonymous customer %      : {missing_pct:.2f}%")

print("\nSample records with missing customer_id:")
display(sales[sales["customer_id"].isna()].head(10))

CUSTOMER ID ROOT CAUSE ANALYSIS
Total sales records       : 641,843
Known customer records    : 482,022
Anonymous customer records: 159,821
Anonymous customer %      : 24.90%

Sample records with missing customer_id:


,date,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct
1,2021-01-01,19,1035,NaN,3,208.57,625.71,Store,15.0
2,2021-01-01,38,1088,NaN,1,17.99,17.99,Website,15.0
6,2021-01-01,2,1043,NaN,3,19.93,59.79,MobileApp,15.0
13,2021-01-01,15,1174,NaN,5,240.85,1204.25,Store,0.0
14,2021-01-01,11,1160,NaN,5,6.18,30.90,Amazon.ae,0.0
15,2021-01-01,32,1135,NaN,2,50.49,100.98,Website,0.0
18,2021-01-01,46,1055,NaN,6,11.14,66.84,Store,0.0
23,2021-01-01,3,1053,NaN,5,11.65,58.25,MobileApp,0.0
25,2021-01-01,25,1074,NaN,1,53.27,53.27,Website,15.0
28,2021-01-01,11,1027,NaN,4,5.14,20.56,MobileApp,15.0


In [73]:
# Duplicate Investigation

duplicate_rows = sales[sales.duplicated(keep=False)].sort_values(
    by=["date", "store_id", "sku_id"]
)

print("=" * 60)
print("DUPLICATE SALES INVESTIGATION")
print("=" * 60)

print(f"Total sales records       : {len(sales):,}")
print(f"Exact duplicate rows      : {sales.duplicated().sum():,}")
print(f"Rows involved in duplicate groups: {len(duplicate_rows):,}")

print("\nSample duplicate groups:")
display(duplicate_rows.head(20))

DUPLICATE SALES INVESTIGATION
Total sales records       : 641,843
Exact duplicate rows      : 45
Rows involved in duplicate groups: 89

Sample duplicate groups:


,date,store_id,sku_id,customer_id,quantity,unit_price,total_value,channel,discount_pct
5463,2021-01-12,28,1061,NaN,2,45.75,91.50,Store,15.0
5783,2021-01-12,28,1061,NaN,2,45.75,91.50,Store,15.0
31125,2021-03-06,10,1037,NaN,1,178.57,178.57,Store,0.0
31306,2021-03-06,10,1037,NaN,1,178.57,178.57,Store,0.0
66354,2021-06-06,10,1145,NaN,1,11.46,11.46,Store,0.0
66573,2021-06-06,10,1145,NaN,1,11.46,11.46,Store,0.0
71206,2021-06-19,13,1024,NaN,2,8.79,17.58,Store,0.0
71332,2021-06-19,13,1024,NaN,2,8.79,17.58,Store,0.0
87236,2021-07-31,27,1069,NaN,5,34.47,172.35,MobileApp,10.0
87384,2021-07-31,27,1069,NaN,5,34.47,172.35,MobileApp,10.0


In [74]:

# Referential Integrity Check

print("=" * 65)
print("REFERENTIAL INTEGRITY CHECK")
print("=" * 65)

# Store IDs in sales that don't exist in store master
invalid_stores = sales.loc[
    ~sales["store_id"].isin(stores["store_id"]),
    "store_id"
].unique()

# SKU IDs in sales that don't exist in SKU master
invalid_skus = sales.loc[
    ~sales["sku_id"].isin(skus["sku_id"]),
    "sku_id"
].unique()

# Customer IDs in sales that don't exist in customer master
# Ignore anonymous customers (NaN)
known_sales_customers = sales.loc[
    sales["customer_id"].notna(),
    "customer_id"
]

invalid_customers = known_sales_customers[
    ~known_sales_customers.isin(customers["customer_id"])
].unique()

print(f"Invalid Store IDs    : {len(invalid_stores):,}")
print(f"Invalid SKU IDs      : {len(invalid_skus):,}")
print(f"Invalid Customer IDs : {len(invalid_customers):,}")

print("\nRESULT")

if (
    len(invalid_stores) == 0
    and len(invalid_skus) == 0
    and len(invalid_customers) == 0
):
    print("PASS - All foreign keys match their master tables.")
else:
    print("WARNING - Referential integrity issues detected.")

REFERENTIAL INTEGRITY CHECK


KeyError: 'customer_id'

In [75]:
# Inspect column names before referential integrity testing

for name, df in datasets.items():
    print(f"\n{name.upper()}")
    print(list(df.columns))


SALES
['date', 'store_id', 'sku_id', 'customer_id', 'quantity', 'unit_price', 'total_value', 'channel', 'discount_pct']

CUSTOMERS
['cust_id', 'age', 'gender', 'city', 'loyalty_segment', 'preferred_channel', 'registration_date']

INVENTORY
['store_id', 'sku_id', 'stock_on_hand', 'reorder_point', 'safety_stock', 'last_restock_date', 'snapshot_date']

PROMOTIONS
['promo_name', 'start_date', 'end_date', 'discount_pct', 'promo_type', 'promo_id']

SKUS
['sku_id', 'sku_name', 'category', 'subcategory', 'unit_price', 'cost_price', 'brand']

STORES
['store_id', 'store_name', 'city', 'store_type', 'opening_date']


In [76]:
# Referential Integrity Check - Corrected Keys

print("=" * 65)
print("REFERENTIAL INTEGRITY CHECK")
print("=" * 65)

# Check Store IDs
invalid_stores = sales.loc[
    ~sales["store_id"].isin(stores["store_id"]),
    "store_id"
].unique()

# Check SKU IDs
invalid_skus = sales.loc[
    ~sales["sku_id"].isin(skus["sku_id"]),
    "sku_id"
].unique()

# Check known Customer IDs
known_customers = sales.loc[
    sales["customer_id"].notna(),
    "customer_id"
]

invalid_customers = known_customers[
    ~known_customers.isin(customers["cust_id"])
].unique()

print(f"Invalid Store IDs    : {len(invalid_stores):,}")
print(f"Invalid SKU IDs      : {len(invalid_skus):,}")
print(f"Invalid Customer IDs : {len(invalid_customers):,}")

print("\nRESULT")

if (
    len(invalid_stores) == 0
    and len(invalid_skus) == 0
    and len(invalid_customers) == 0
):
    print("PASS - All foreign keys match their master tables.")
else:
    print("WARNING - Referential integrity issues detected.")

REFERENTIAL INTEGRITY CHECK
Invalid Store IDs    : 0
Invalid SKU IDs      : 0
Invalid Customer IDs : 0

RESULT
PASS - All foreign keys match their master tables.


In [77]:
# STEP 7 - Business Validation

sales["date"] = pd.to_datetime(sales["date"])

print("=" * 65)
print("BUSINESS VALIDATION - SALES")
print("=" * 65)

print(f"Date range              : {sales['date'].min().date()} to {sales['date'].max().date()}")
print(f"Negative quantities     : {(sales['quantity'] < 0).sum():,}")
print(f"Zero quantities         : {(sales['quantity'] == 0).sum():,}")
print(f"Negative unit prices    : {(sales['unit_price'] < 0).sum():,}")
print(f"Zero unit prices        : {(sales['unit_price'] == 0).sum():,}")
print(f"Negative total values   : {(sales['total_value'] < 0).sum():,}")
print(f"Zero total values       : {(sales['total_value'] == 0).sum():,}")
print(f"Discount below 0%       : {(sales['discount_pct'] < 0).sum():,}")
print(f"Discount above 100%     : {(sales['discount_pct'] > 100).sum():,}")

print("\nDiscount values:")
print(sorted(sales["discount_pct"].dropna().unique()))

# Validate sales arithmetic
sales["calculated_value"] = (
    sales["quantity"] * sales["unit_price"]
).round(2)

value_mismatch = (
    (sales["calculated_value"] - sales["total_value"]).abs() > 0.01
)

print(f"\nSales value mismatches  : {value_mismatch.sum():,}")
print(f"Mismatch rate           : {value_mismatch.mean()*100:.4f}%")

BUSINESS VALIDATION - SALES
Date range              : 2021-01-01 to 2025-10-31
Negative quantities     : 0
Zero quantities         : 0
Negative unit prices    : 0
Zero unit prices        : 0
Negative total values   : 0
Zero total values       : 0
Discount below 0%       : 0
Discount above 100%     : 0

Discount values:
[np.float64(0.0), np.float64(10.0), np.float64(15.0), np.float64(20.0), np.float64(25.0), np.float64(30.0), np.float64(35.0)]

Sales value mismatches  : 0
Mismatch rate           : 0.0000%


In [78]:
# STEP 8 - Profitability Validation

sales_profit = sales.merge(
    skus[["sku_id", "sku_name", "category", "cost_price"]],
    on="sku_id",
    how="left"
)

# Calculate profitability
sales_profit["cogs"] = (
    sales_profit["quantity"] * sales_profit["cost_price"]
)

sales_profit["gross_profit"] = (
    sales_profit["total_value"] - sales_profit["cogs"]
)

sales_profit["gross_margin_pct"] = (
    sales_profit["gross_profit"] /
    sales_profit["total_value"] * 100
)

print("=" * 65)
print("PROFITABILITY VALIDATION")
print("=" * 65)

print(f"Sales rows               : {len(sales_profit):,}")
print(f"Missing cost prices      : {sales_profit['cost_price'].isna().sum():,}")
print(f"Negative cost prices     : {(sales_profit['cost_price'] < 0).sum():,}")
print(f"Negative gross profit    : {(sales_profit['gross_profit'] < 0).sum():,}")

print("\nENTERPRISE PROFITABILITY")

print(f"Total Revenue            : ${sales_profit['total_value'].sum():,.2f}")
print(f"Total COGS               : ${sales_profit['cogs'].sum():,.2f}")
print(f"Total Gross Profit       : ${sales_profit['gross_profit'].sum():,.2f}")

overall_margin = (
    sales_profit["gross_profit"].sum() /
    sales_profit["total_value"].sum() * 100
)

print(f"Overall Gross Margin     : {overall_margin:.2f}%")

PROFITABILITY VALIDATION
Sales rows               : 641,843
Missing cost prices      : 0
Negative cost prices     : 0
Negative gross profit    : 10,291

ENTERPRISE PROFITABILITY
Total Revenue            : $73,214,931.30
Total COGS               : $52,116,235.58
Total Gross Profit       : $21,098,695.72
Overall Gross Margin     : 28.82%


In [79]:
# STEP 9 - Investigate Loss-Making Sales

loss_sales = sales_profit[
    sales_profit["gross_profit"] < 0
].copy()

print("=" * 70)
print("LOSS-MAKING SALES INVESTIGATION")
print("=" * 70)

print(f"Loss-making rows        : {len(loss_sales):,}")
print(
    f"Loss-making rate        : "
    f"{len(loss_sales) / len(sales_profit) * 100:.2f}%"
)

print(
    f"Revenue from loss sales : "
    f"${loss_sales['total_value'].sum():,.2f}"
)

print(
    f"COGS from loss sales    : "
    f"${loss_sales['cogs'].sum():,.2f}"
)

print(
    f"Gross loss              : "
    f"${loss_sales['gross_profit'].sum():,.2f}"
)

print("\nLOSS-MAKING SALES BY DISCOUNT")

loss_by_discount = (
    loss_sales
    .groupby("discount_pct")
    .agg(
        rows=("sku_id", "count"),
        revenue=("total_value", "sum"),
        gross_profit=("gross_profit", "sum")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(loss_by_discount)

print("\nSAMPLE LOSS-MAKING SALES")

display(
    loss_sales[
        [
            "date",
            "store_id",
            "sku_id",
            "sku_name",
            "quantity",
            "unit_price",
            "cost_price",
            "discount_pct",
            "total_value",
            "gross_profit",
            "gross_margin_pct"
        ]
    ].head(20)
)

LOSS-MAKING SALES INVESTIGATION
Loss-making rows        : 10,291
Loss-making rate        : 1.60%
Revenue from loss sales : $1,066,973.75
COGS from loss sales    : $1,121,992.80
Gross loss              : $-55,019.05

LOSS-MAKING SALES BY DISCOUNT


,discount_pct,rows,revenue,gross_profit
0,25.0,5680,627126.40,-22820.71
1,30.0,3413,329664.48,-21527.71
2,35.0,1198,110182.87,-10670.63



SAMPLE LOSS-MAKING SALES


,date,store_id,sku_id,sku_name,quantity,unit_price,cost_price,discount_pct,total_value,gross_profit,gross_margin_pct
42049,2021-04-13,10,1095,Snacks_Chocolates_1095,2,15.69,16.47,25.0,31.38,-1.56,-4.971319
42061,2021-04-13,43,1068,Electronics_Chargers_1068,2,71.32,74.43,25.0,142.64,-6.22,-4.360628
42065,2021-04-13,38,1151,Electronics_Cables_1151,5,107.05,110.85,25.0,535.25,-19.00,-3.549743
42083,2021-04-13,38,1005,Household_Paper Products_1005,6,13.27,13.68,25.0,79.62,-2.46,-3.089676
42088,2021-04-13,5,1096,Electronics_Mobile Accessories_1096,1,84.97,88.73,25.0,84.97,-3.76,-4.425091
42108,2021-04-13,1,1185,Personal Care_Toothpaste_1185,2,58.04,60.84,25.0,116.08,-5.60,-4.824259
42110,2021-04-13,12,1084,Household_Paper Products_1084,1,35.92,36.19,25.0,35.92,-0.27,-0.751670
42124,2021-04-13,38,1096,Electronics_Mobile Accessories_1096,5,84.97,88.73,25.0,424.85,-18.80,-4.425091
42129,2021-04-13,24,1005,Household_Paper Products_1005,6,13.27,13.68,25.0,79.62,-2.46,-3.089676
42133,2021-04-13,43,1095,Snacks_Chocolates_1095,4,15.69,16.47,25.0,62.76,-3.12,-4.971319


In [80]:
# STEP 10 - Validate Discount Pricing Logic

pricing_check = sales.merge(
    skus[["sku_id", "unit_price", "cost_price"]],
    on="sku_id",
    how="left",
    suffixes=("_sale", "_list")
)

# Expected selling price after discount
pricing_check["expected_price"] = (
    pricing_check["unit_price_list"] *
    (1 - pricing_check["discount_pct"] / 100)
).round(2)

pricing_check["price_difference"] = (
    pricing_check["unit_price_sale"] -
    pricing_check["expected_price"]
).round(2)

price_match = pricing_check["price_difference"].abs() <= 0.01

print("=" * 70)
print("DISCOUNT PRICING LOGIC VALIDATION")
print("=" * 70)

print(f"Total rows checked       : {len(pricing_check):,}")
print(f"Price logic matches      : {price_match.sum():,}")
print(f"Price logic mismatches   : {(~price_match).sum():,}")
print(f"Match rate               : {price_match.mean()*100:.2f}%")

print("\nSAMPLE PRICING VALIDATION")

display(
    pricing_check[
        [
            "sku_id",
            "unit_price_list",
            "discount_pct",
            "expected_price",
            "unit_price_sale",
            "cost_price",
            "price_difference"
        ]
    ].head(20)
)

DISCOUNT PRICING LOGIC VALIDATION
Total rows checked       : 641,843
Price logic matches      : 641,843
Price logic mismatches   : 0
Match rate               : 100.00%

SAMPLE PRICING VALIDATION


,sku_id,unit_price_list,discount_pct,expected_price,unit_price_sale,cost_price,price_difference
0,1124,23.62,15.0,20.08,20.08,13.62,0.0
1,1035,245.38,15.0,208.57,208.57,139.58,0.0
2,1088,21.16,15.0,17.99,17.99,15.32,0.0
3,1164,196.24,0.0,196.24,196.24,126.30,0.0
4,1093,7.98,0.0,7.98,7.98,5.29,0.0
5,1067,49.57,15.0,42.13,42.13,32.38,0.0
6,1043,23.45,15.0,19.93,19.93,15.63,0.0
7,1008,43.94,15.0,37.35,37.35,31.77,0.0
8,1121,31.88,0.0,31.88,31.88,25.39,0.0
9,1078,179.73,0.0,179.73,179.73,102.12,0.0


In [81]:
# STEP 11 - SKU-Level Maximum Safe Discount / Margin Guardrail

sku_guardrail = skus.copy()

# Break-even discount:
# list_price * (1 - discount%) = cost_price
sku_guardrail["max_safe_discount_pct"] = (
    (1 - (sku_guardrail["cost_price"] / sku_guardrail["unit_price"])) * 100
)

sku_guardrail["max_safe_discount_pct"] = (
    sku_guardrail["max_safe_discount_pct"]
    .clip(lower=0)
    .round(2)
)

# Current gross margin before discount
sku_guardrail["base_margin_pct"] = (
    (sku_guardrail["unit_price"] - sku_guardrail["cost_price"])
    / sku_guardrail["unit_price"] * 100
).round(2)

# Flag SKUs that cannot safely support common discount levels
for d in [15, 20, 25, 30, 35]:
    sku_guardrail[f"safe_at_{d}pct"] = (
        sku_guardrail["max_safe_discount_pct"] >= d
    )

print("=" * 75)
print("SKU-LEVEL DISCOUNT GUARDRAIL")
print("=" * 75)

print(f"Total SKUs analysed      : {len(sku_guardrail):,}")
print(f"Avg base gross margin    : {sku_guardrail['base_margin_pct'].mean():.2f}%")
print(f"Avg max safe discount    : {sku_guardrail['max_safe_discount_pct'].mean():.2f}%")

print("\nSKUs SAFE AT EACH DISCOUNT LEVEL")

for d in [15, 20, 25, 30, 35]:
    safe_count = sku_guardrail[f"safe_at_{d}pct"].sum()
    unsafe_count = len(sku_guardrail) - safe_count

    print(
        f"{d:>2}% discount | "
        f"Safe: {safe_count:>3} | "
        f"Unsafe: {unsafe_count:>3}"
    )

print("\n10 MOST VULNERABLE SKUs")

display(
    sku_guardrail[
        [
            "sku_id",
            "sku_name",
            "category",
            "unit_price",
            "cost_price",
            "base_margin_pct",
            "max_safe_discount_pct"
        ]
    ]
    .sort_values("max_safe_discount_pct")
    .head(10)
)

SKU-LEVEL DISCOUNT GUARDRAIL
Total SKUs analysed      : 200
Avg base gross margin    : 33.44%
Avg max safe discount    : 33.44%

SKUs SAFE AT EACH DISCOUNT LEVEL
15% discount | Safe: 200 | Unsafe:   0
20% discount | Safe: 200 | Unsafe:   0
25% discount | Safe: 169 | Unsafe:  31
30% discount | Safe: 134 | Unsafe:  66
35% discount | Safe:  88 | Unsafe: 112

10 MOST VULNERABLE SKUs


,sku_id,sku_name,category,unit_price,cost_price,base_margin_pct,max_safe_discount_pct
10,1011,Personal Care_Shampoo_1011,Personal Care,20.37,16.28,20.08,20.08
120,1121,Grocery_Rice_1121,Grocery,31.88,25.39,20.36,20.36
136,1137,Snacks_Nuts_1137,Snacks,6.72,5.32,20.83,20.83
85,1086,Grocery_Rice_1086,Grocery,15.06,11.91,20.92,20.92
71,1072,Electronics_Chargers_1072,Electronics,238.26,187.67,21.23,21.23
94,1095,Snacks_Chocolates_1095,Snacks,20.92,16.47,21.27,21.27
184,1185,Personal Care_Toothpaste_1185,Personal Care,77.39,60.84,21.39,21.39
95,1096,Electronics_Mobile Accessories_1096,Electronics,113.29,88.73,21.68,21.68
138,1139,Dairy_Cream_1139,Dairy,22.71,17.78,21.71,21.71
67,1068,Electronics_Chargers_1068,Electronics,95.09,74.43,21.73,21.73


In [82]:
# STEP 12 - Quantify Preventable Margin Leakage

guardrail_sales = sales_profit.merge(
    sku_guardrail[
        ["sku_id", "unit_price", "max_safe_discount_pct"]
    ],
    on="sku_id",
    how="left",
    suffixes=("_sale", "_list")
)

# Identify transactions exceeding SKU break-even discount
guardrail_sales["guardrail_breach"] = (
    guardrail_sales["discount_pct"] >
    guardrail_sales["max_safe_discount_pct"]
)

breaches = guardrail_sales[
    guardrail_sales["guardrail_breach"]
].copy()

# Actual loss on transactions that breached guardrail
breaches["preventable_loss"] = (
    -breaches["gross_profit"].clip(upper=0)
)

total_preventable_loss = breaches["preventable_loss"].sum()

print("=" * 75)
print("DISCOUNT GUARDRAIL - FINANCIAL IMPACT")
print("=" * 75)

print(f"Total sales rows             : {len(guardrail_sales):,}")
print(f"Guardrail breaches           : {len(breaches):,}")
print(
    f"Breach rate                  : "
    f"{len(breaches)/len(guardrail_sales)*100:.2f}%"
)

print(f"\nRevenue exposed              : ${breaches['total_value'].sum():,.2f}")
print(f"Preventable gross loss       : ${total_preventable_loss:,.2f}")

print("\nFINANCIAL IMPACT BY DISCOUNT")

impact_discount = (
    breaches
    .groupby("discount_pct")
    .agg(
        transactions=("sku_id", "count"),
        revenue_exposed=("total_value", "sum"),
        preventable_loss=("preventable_loss", "sum")
    )
    .reset_index()
)

display(impact_discount)

print("\nTOP 10 SKUs BY PREVENTABLE LOSS")

impact_sku = (
    breaches
    .groupby(["sku_id", "sku_name"])
    .agg(
        transactions=("sku_id", "count"),
        revenue_exposed=("total_value", "sum"),
        preventable_loss=("preventable_loss", "sum")
    )
    .reset_index()
    .sort_values("preventable_loss", ascending=False)
)

display(impact_sku.head(10))

DISCOUNT GUARDRAIL - FINANCIAL IMPACT
Total sales rows             : 641,843
Guardrail breaches           : 10,314
Breach rate                  : 1.61%

Revenue exposed              : $1,063,330.53
Preventable gross loss       : $55,018.49

FINANCIAL IMPACT BY DISCOUNT


,discount_pct,transactions,revenue_exposed,preventable_loss
0,25.0,5680,627126.40,22820.71
1,30.0,3452,331569.18,21527.71
2,35.0,1182,104634.95,10670.07



TOP 10 SKUs BY PREVENTABLE LOSS


,sku_id,sku_name,transactions,revenue_exposed,preventable_loss
34,1072,Electronics_Chargers_1072,263,141219.60,9667.08
45,1096,Electronics_Mobile Accessories_1096,263,69861.19,4849.47
32,1068,Electronics_Chargers_1068,270,58272.11,3802.51
77,1151,Electronics_Cables_1151,227,77018.43,3791.22
91,1172,Electronics_Chargers_1172,227,59102.52,3265.52
101,1185,Personal Care_Toothpaste_1185,235,44056.08,3034.08
58,1121,Grocery_Rice_1121,287,22436.09,1785.97
4,1007,Personal Care_Shampoo_1007,250,38212.86,1676.39
13,1023,Personal Care_Toothpaste_1023,226,32039.18,1535.99
46,1098,Electronics_Mobile Accessories_1098,72,16657.59,1129.63


In [83]:
# STEP 12A - Row Count Reconciliation

print("=" * 65)
print("ROW COUNT RECONCILIATION")
print("=" * 65)

print(f"sales rows          : {len(sales):,}")
print(f"sales_profit rows   : {len(sales_profit):,}")
print(f"sku_guardrail rows  : {len(sku_guardrail):,}")
print(f"guardrail_sales rows: {len(guardrail_sales):,}")

print("\nUnique SKUs")
print(f"sales          : {sales['sku_id'].nunique():,}")
print(f"sales_profit   : {sales_profit['sku_id'].nunique():,}")
print(f"sku_guardrail  : {sku_guardrail['sku_id'].nunique():,}")

ROW COUNT RECONCILIATION
sales rows          : 641,843
sales_profit rows   : 641,843
sku_guardrail rows  : 200
guardrail_sales rows: 641,843

Unique SKUs
sales          : 200
sales_profit   : 200
sku_guardrail  : 200


In [84]:
# STEP 12B - Reload Original Sales Data Safely

sales_raw = pd.read_csv("bm_sales.csv")

print("=" * 65)
print("RAW SALES RECOVERY CHECK")
print("=" * 65)

print(f"CSV original rows : {len(sales_raw):,}")
print(f"Current sales rows: {len(sales):,}")
print(f"Difference        : {len(sales_raw) - len(sales):,}")

RAW SALES RECOVERY CHECK
CSV original rows : 641,843
Current sales rows: 641,843
Difference        : 0


In [85]:
# STEP 12C - Restore Master Sales Dataset

sales = sales_raw.copy()

# Ensure correct date datatype
sales["date"] = pd.to_datetime(sales["date"])

print("=" * 65)
print("MASTER SALES RESTORED")
print("=" * 65)

print(f"Original CSV rows : {len(sales_raw):,}")
print(f"Restored sales    : {len(sales):,}")
print(f"Unique SKUs       : {sales['sku_id'].nunique():,}")
print(f"Unique stores     : {sales['store_id'].nunique():,}")

if len(sales) == len(sales_raw):
    print("\nPASS - Full sales dataset successfully restored.")
else:
    print("\nWARNING - Row count mismatch.")

MASTER SALES RESTORED
Original CSV rows : 641,843
Restored sales    : 641,843
Unique SKUs       : 200
Unique stores     : 50

PASS - Full sales dataset successfully restored.


In [86]:
# STEP 12D - Rebuild Profitability Using Full Sales Dataset

sales_profit = sales.merge(
    skus[
        ["sku_id", "sku_name", "category", "cost_price"]
    ],
    on="sku_id",
    how="left"
)

sales_profit["cogs"] = (
    sales_profit["quantity"] *
    sales_profit["cost_price"]
)

sales_profit["gross_profit"] = (
    sales_profit["total_value"] -
    sales_profit["cogs"]
)

sales_profit["gross_margin_pct"] = (
    sales_profit["gross_profit"] /
    sales_profit["total_value"] * 100
)

print("=" * 70)
print("FULL PROFITABILITY REBUILD")
print("=" * 70)

print(f"Input sales rows       : {len(sales):,}")
print(f"Output profitability   : {len(sales_profit):,}")
print(f"Missing cost price     : {sales_profit['cost_price'].isna().sum():,}")

print("\nFINANCIAL RECONCILIATION")

print(
    f"Revenue     : "
    f"${sales_profit['total_value'].sum():,.2f}"
)

print(
    f"COGS        : "
    f"${sales_profit['cogs'].sum():,.2f}"
)

print(
    f"Gross Profit: "
    f"${sales_profit['gross_profit'].sum():,.2f}"
)

overall_margin = (
    sales_profit["gross_profit"].sum() /
    sales_profit["total_value"].sum() * 100
)

print(
    f"Gross Margin: "
    f"{overall_margin:.2f}%"
)

print(
    f"Loss-making rows: "
    f"{(sales_profit['gross_profit'] < 0).sum():,}"
)

if len(sales_profit) == len(sales):
    print("\nPASS - Row count preserved after profitability merge.")
else:
    print("\nWARNING - Row count changed during merge.")

FULL PROFITABILITY REBUILD
Input sales rows       : 641,843
Output profitability   : 641,843
Missing cost price     : 0

FINANCIAL RECONCILIATION
Revenue     : $73,214,931.30
COGS        : $52,116,235.58
Gross Profit: $21,098,695.72
Gross Margin: 28.82%
Loss-making rows: 10,291

PASS - Row count preserved after profitability merge.


In [87]:
# STEP 12E - Final Discount Guardrail Financial Impact

guardrail_sales = sales_profit.merge(
    sku_guardrail[
        ["sku_id", "max_safe_discount_pct"]
    ],
    on="sku_id",
    how="left"
)

# Guardrail breach
guardrail_sales["guardrail_breach"] = (
    guardrail_sales["discount_pct"] >
    guardrail_sales["max_safe_discount_pct"]
)

# Only breaches that actually produced negative gross profit
loss_breaches = guardrail_sales[
    (guardrail_sales["guardrail_breach"]) &
    (guardrail_sales["gross_profit"] < 0)
].copy()

loss_breaches["preventable_loss"] = (
    -loss_breaches["gross_profit"]
)

print("=" * 75)
print("FINAL DISCOUNT GUARDRAIL FINANCIAL IMPACT")
print("=" * 75)

print(f"Input sales rows          : {len(sales_profit):,}")
print(f"Rows after guardrail merge: {len(guardrail_sales):,}")
print(f"Loss-making breaches      : {len(loss_breaches):,}")

print(
    f"Loss-making breach rate   : "
    f"{len(loss_breaches)/len(guardrail_sales)*100:.2f}%"
)

print(
    f"\nRevenue exposed           : "
    f"${loss_breaches['total_value'].sum():,.2f}"
)

print(
    f"Preventable gross loss    : "
    f"${loss_breaches['preventable_loss'].sum():,.2f}"
)

print("\nIMPACT BY DISCOUNT LEVEL")

impact_by_discount = (
    loss_breaches
    .groupby("discount_pct")
    .agg(
        rows=("sku_id", "count"),
        revenue_exposed=("total_value", "sum"),
        preventable_loss=("preventable_loss", "sum")
    )
    .reset_index()
)

display(impact_by_discount)

print("\nTOP 10 SKUs BY MARGIN LEAKAGE")

top_loss_skus = (
    loss_breaches
    .groupby(["sku_id", "sku_name"])
    .agg(
        rows=("sku_id", "count"),
        revenue_exposed=("total_value", "sum"),
        preventable_loss=("preventable_loss", "sum")
    )
    .reset_index()
    .sort_values("preventable_loss", ascending=False)
)

display(top_loss_skus.head(10))

# Final reconciliation check
if len(guardrail_sales) == len(sales_profit):
    print("\nPASS - No rows lost during guardrail analysis.")
else:
    print("\nWARNING - Row count changed during guardrail merge.")

FINAL DISCOUNT GUARDRAIL FINANCIAL IMPACT
Input sales rows          : 641,843
Rows after guardrail merge: 641,843
Loss-making breaches      : 10,275
Loss-making breach rate   : 1.60%

Revenue exposed           : $1,061,425.83
Preventable gross loss    : $55,018.49

IMPACT BY DISCOUNT LEVEL


,discount_pct,rows,revenue_exposed,preventable_loss
0,25.0,5680,627126.40,22820.71
1,30.0,3413,329664.48,21527.71
2,35.0,1182,104634.95,10670.07



TOP 10 SKUs BY MARGIN LEAKAGE


,sku_id,sku_name,rows,revenue_exposed,preventable_loss
34,1072,Electronics_Chargers_1072,263,141219.60,9667.08
45,1096,Electronics_Mobile Accessories_1096,263,69861.19,4849.47
32,1068,Electronics_Chargers_1068,270,58272.11,3802.51
77,1151,Electronics_Cables_1151,227,77018.43,3791.22
91,1172,Electronics_Chargers_1172,227,59102.52,3265.52
101,1185,Personal Care_Toothpaste_1185,235,44056.08,3034.08
58,1121,Grocery_Rice_1121,287,22436.09,1785.97
4,1007,Personal Care_Shampoo_1007,250,38212.86,1676.39
13,1023,Personal Care_Toothpaste_1023,226,32039.18,1535.99
46,1098,Electronics_Mobile Accessories_1098,72,16657.59,1129.63



PASS - No rows lost during guardrail analysis.


In [88]:
# STEP 13A - Inventory Health Audit

# Ensure dates are datetime
inventory["snapshot_date"] = pd.to_datetime(inventory["snapshot_date"])
inventory["last_restock_date"] = pd.to_datetime(inventory["last_restock_date"])

print("=" * 70)
print("INVENTORY HEALTH AUDIT")
print("=" * 70)

print(f"Inventory rows          : {len(inventory):,}")
print(f"Unique stores           : {inventory['store_id'].nunique():,}")
print(f"Unique SKUs             : {inventory['sku_id'].nunique():,}")
print(
    f"Snapshot range          : "
    f"{inventory['snapshot_date'].min().date()} "
    f"to {inventory['snapshot_date'].max().date()}"
)

print("\nDATA QUALITY")

print(
    f"Missing stock_on_hand   : "
    f"{inventory['stock_on_hand'].isna().sum():,}"
)

print(
    f"Missing reorder_point   : "
    f"{inventory['reorder_point'].isna().sum():,}"
)

print(
    f"Missing safety_stock    : "
    f"{inventory['safety_stock'].isna().sum():,}"
)

print(
    f"Negative stock          : "
    f"{(inventory['stock_on_hand'] < 0).sum():,}"
)

# Inventory conditions
stockout = inventory["stock_on_hand"] == 0

below_safety = (
    inventory["stock_on_hand"] <
    inventory["safety_stock"]
)

reorder_required = (
    inventory["stock_on_hand"] <=
    inventory["reorder_point"]
)

print("\nINVENTORY RISK")

print(
    f"Stockout records        : "
    f"{stockout.sum():,} "
    f"({stockout.mean()*100:.2f}%)"
)

print(
    f"Below safety stock      : "
    f"{below_safety.sum():,} "
    f"({below_safety.mean()*100:.2f}%)"
)

print(
    f"At/below reorder point  : "
    f"{reorder_required.sum():,} "
    f"({reorder_required.mean()*100:.2f}%)"
)

print("\nSAMPLE INVENTORY")
display(inventory.head(10))

INVENTORY HEALTH AUDIT
Inventory rows          : 8,735
Unique stores           : 50
Unique SKUs             : 200
Snapshot range          : 2025-10-31 to 2025-10-31

DATA QUALITY
Missing stock_on_hand   : 0
Missing reorder_point   : 0
Missing safety_stock    : 0
Negative stock          : 0

INVENTORY RISK
Stockout records        : 0 (0.00%)
Below safety stock      : 0 (0.00%)
At/below reorder point  : 0 (0.00%)

SAMPLE INVENTORY


,store_id,sku_id,stock_on_hand,reorder_point,safety_stock,last_restock_date,snapshot_date
0,9,1101,300,92,46,2025-10-18,2025-10-31
1,9,1021,425,162,81,2025-10-21,2025-10-31
2,9,1200,252,85,42,2025-09-14,2025-10-31
3,9,1041,220,87,43,2025-10-13,2025-10-31
4,9,1092,170,80,40,2025-09-12,2025-10-31
5,9,1094,262,113,56,2025-08-28,2025-10-31
6,9,1143,293,93,46,2025-08-30,2025-10-31
7,9,1111,271,89,44,2025-10-16,2025-10-31
8,9,1119,342,165,82,2025-10-25,2025-10-31
9,9,1025,126,54,27,2025-10-14,2025-10-31


In [89]:
# STEP 13B - Sales Velocity & Days of Supply

# Use recent 30-day sales window before inventory snapshot
snapshot_date = inventory["snapshot_date"].max()
window_start = snapshot_date - pd.Timedelta(days=29)

recent_sales = sales[
    (sales["date"] >= window_start) &
    (sales["date"] <= snapshot_date)
].copy()

print("=" * 70)
print("30-DAY SALES VELOCITY")
print("=" * 70)

print(f"Inventory snapshot : {snapshot_date.date()}")
print(f"Sales window       : {window_start.date()} to {snapshot_date.date()}")
print(f"Sales rows used    : {len(recent_sales):,}")

# Sales velocity by store + SKU
velocity = (
    recent_sales
    .groupby(["store_id", "sku_id"])
    .agg(
        units_sold_30d=("quantity", "sum"),
        revenue_30d=("total_value", "sum")
    )
    .reset_index()
)

velocity["avg_daily_units"] = (
    velocity["units_sold_30d"] / 30
)

# Merge with inventory
inventory_velocity = inventory.merge(
    velocity,
    on=["store_id", "sku_id"],
    how="left"
)

# Products with no sales in last 30 days
inventory_velocity[
    ["units_sold_30d", "revenue_30d", "avg_daily_units"]
] = inventory_velocity[
    ["units_sold_30d", "revenue_30d", "avg_daily_units"]
].fillna(0)

# Days of Supply
inventory_velocity["days_of_supply"] = np.where(
    inventory_velocity["avg_daily_units"] > 0,
    inventory_velocity["stock_on_hand"] /
    inventory_velocity["avg_daily_units"],
    np.nan
)

print("\nVELOCITY COVERAGE")

print(
    f"Inventory combinations : "
    f"{len(inventory_velocity):,}"
)

print(
    f"With recent sales       : "
    f"{(inventory_velocity['avg_daily_units'] > 0).sum():,}"
)

print(
    f"No sales in 30 days     : "
    f"{(inventory_velocity['avg_daily_units'] == 0).sum():,}"
)

print("\nDAYS OF SUPPLY")

valid_dos = inventory_velocity["days_of_supply"].dropna()

print(f"Average DOS : {valid_dos.mean():.2f} days")
print(f"Median DOS  : {valid_dos.median():.2f} days")
print(f"Minimum DOS : {valid_dos.min():.2f} days")
print(f"Maximum DOS : {valid_dos.max():.2f} days")

print("\nSAMPLE")
display(
    inventory_velocity[
        [
            "store_id",
            "sku_id",
            "stock_on_hand",
            "units_sold_30d",
            "avg_daily_units",
            "days_of_supply"
        ]
    ].head(20)
)

30-DAY SALES VELOCITY
Inventory snapshot : 2025-10-31
Sales window       : 2025-10-02 to 2025-10-31
Sales rows used    : 8,073

VELOCITY COVERAGE
Inventory combinations : 8,735
With recent sales       : 4,848
No sales in 30 days     : 3,887

DAYS OF SUPPLY
Average DOS : 2582.38 days
Median DOS  : 1890.00 days
Minimum DOS : 138.75 days
Maximum DOS : 13440.00 days

SAMPLE


,store_id,sku_id,stock_on_hand,units_sold_30d,avg_daily_units,days_of_supply
0,9,1101,300,0.0,0.000000,NaN
1,9,1021,425,0.0,0.000000,NaN
2,9,1200,252,2.0,0.066667,3780.0
3,9,1041,220,4.0,0.133333,1650.0
4,9,1092,170,0.0,0.000000,NaN
5,9,1094,262,0.0,0.000000,NaN
6,9,1143,293,0.0,0.000000,NaN
7,9,1111,271,0.0,0.000000,NaN
8,9,1119,342,0.0,0.000000,NaN
9,9,1025,126,0.0,0.000000,NaN


In [90]:
# STEP 13C - Inventory Risk Classification

inventory_risk = inventory_velocity.copy()

# Classification based on recent sales velocity
def classify_inventory(row):

    if row["units_sold_30d"] == 0:
        return "No Sales - 30 Days"

    elif row["days_of_supply"] > 365:
        return "Severe Overstock"

    elif row["days_of_supply"] > 180:
        return "Overstock"

    elif row["days_of_supply"] > 90:
        return "High Stock"

    else:
        return "Healthy"

inventory_risk["inventory_status"] = inventory_risk.apply(
    classify_inventory,
    axis=1
)

risk_summary = (
    inventory_risk
    .groupby("inventory_status")
    .agg(
        sku_store_combinations=("sku_id", "size"),
        stock_units=("stock_on_hand", "sum")
    )
    .reset_index()
)

risk_summary["pct_inventory_records"] = (
    risk_summary["sku_store_combinations"] /
    len(inventory_risk) * 100
).round(2)

print("=" * 70)
print("INVENTORY RISK CLASSIFICATION")
print("=" * 70)

print(f"Total inventory combinations : {len(inventory_risk):,}")

print(
    f"No-sales combinations       : "
    f"{(inventory_risk['units_sold_30d'] == 0).sum():,}"
)

print(
    f"Severe overstock (>365 DOS) : "
    f"{(inventory_risk['days_of_supply'] > 365).sum():,}"
)

print(
    f"Overstock (>180 DOS)        : "
    f"{((inventory_risk['days_of_supply'] > 180) &
       (inventory_risk['days_of_supply'] <= 365)).sum():,}"
)

print(
    f"Healthy / <=90 DOS          : "
    f"{((inventory_risk['days_of_supply'] <= 90)).sum():,}"
)

print("\nRISK DISTRIBUTION")
display(
    risk_summary.sort_values(
        "sku_store_combinations",
        ascending=False
    )
)

INVENTORY RISK CLASSIFICATION
Total inventory combinations : 8,735
No-sales combinations       : 3,887
Severe overstock (>365 DOS) : 4,775
Overstock (>180 DOS)        : 68
Healthy / <=90 DOS          : 0

RISK DISTRIBUTION


,inventory_status,sku_store_combinations,stock_units,pct_inventory_records
3,Severe Overstock,4775,821002,54.67
1,No Sales - 30 Days,3887,665813,44.50
2,Overstock,68,4561,0.78
0,High Stock,5,247,0.06


In [91]:
# STEP 13D - Quantify Inventory Working Capital Exposure

# Bring SKU cost into inventory analysis
inventory_value = inventory_risk.merge(
    skus[
        ["sku_id", "sku_name", "category", "cost_price"]
    ],
    on="sku_id",
    how="left"
)

# Inventory value at cost
inventory_value["inventory_cost_value"] = (
    inventory_value["stock_on_hand"] *
    inventory_value["cost_price"]
)

print("=" * 70)
print("INVENTORY WORKING CAPITAL EXPOSURE")
print("=" * 70)

print(f"Inventory records       : {len(inventory_value):,}")
print(
    f"Missing cost prices     : "
    f"{inventory_value['cost_price'].isna().sum():,}"
)

print(
    f"Total stock units       : "
    f"{inventory_value['stock_on_hand'].sum():,.0f}"
)

print(
    f"Total inventory value   : "
    f"${inventory_value['inventory_cost_value'].sum():,.2f}"
)

# Summary by inventory status
capital_summary = (
    inventory_value
    .groupby("inventory_status")
    .agg(
        records=("sku_id", "size"),
        stock_units=("stock_on_hand", "sum"),
        inventory_value=("inventory_cost_value", "sum")
    )
    .reset_index()
)

total_inventory_value = inventory_value[
    "inventory_cost_value"
].sum()

capital_summary["pct_inventory_value"] = (
    capital_summary["inventory_value"] /
    total_inventory_value * 100
).round(2)

print("\nWORKING CAPITAL BY INVENTORY STATUS")

display(
    capital_summary.sort_values(
        "inventory_value",
        ascending=False
    )
)

# Potential trapped capital:
# No sales + Overstock + Severe Overstock
risk_statuses = [
    "No Sales - 30 Days",
    "Overstock",
    "Severe Overstock"
]

trapped_inventory = inventory_value[
    inventory_value["inventory_status"].isin(risk_statuses)
].copy()

trapped_value = trapped_inventory[
    "inventory_cost_value"
].sum()

print("\n" + "=" * 70)
print("POTENTIAL WORKING CAPITAL EXPOSURE")
print("=" * 70)

print(
    f"At-risk inventory value : "
    f"${trapped_value:,.2f}"
)

print(
    f"% of inventory value    : "
    f"{trapped_value / total_inventory_value * 100:.2f}%"
)

print("\nTOP 10 STORE-SKU INVENTORY EXPOSURES")

display(
    trapped_inventory[
        [
            "store_id",
            "sku_id",
            "sku_name",
            "category",
            "stock_on_hand",
            "units_sold_30d",
            "days_of_supply",
            "inventory_status",
            "inventory_cost_value"
        ]
    ]
    .sort_values(
        "inventory_cost_value",
        ascending=False
    )
    .head(10)
)

INVENTORY WORKING CAPITAL EXPOSURE
Inventory records       : 8,735
Missing cost prices     : 0
Total stock units       : 1,491,623
Total inventory value   : $38,914,341.92

WORKING CAPITAL BY INVENTORY STATUS


,inventory_status,records,stock_units,inventory_value,pct_inventory_value
3,Severe Overstock,4775,821002,20904980.95,53.72
1,No Sales - 30 Days,3887,665813,17766113.91,45.65
2,Overstock,68,4561,232653.70,0.60
0,High Stock,5,247,10593.36,0.03



POTENTIAL WORKING CAPITAL EXPOSURE
At-risk inventory value : $38,903,748.56
% of inventory value    : 99.97%

TOP 10 STORE-SKU INVENTORY EXPOSURES


,store_id,sku_id,sku_name,category,stock_on_hand,units_sold_30d,days_of_supply,inventory_status,inventory_cost_value
399,11,1072,Electronics_Chargers_1072,Electronics,203,0.0,NaN,No Sales - 30 Days,38097.01
2317,22,1072,Electronics_Chargers_1072,Electronics,199,0.0,NaN,No Sales - 30 Days,37346.33
8569,8,1072,Electronics_Chargers_1072,Electronics,191,0.0,NaN,No Sales - 30 Days,35844.97
4147,30,1072,Electronics_Chargers_1072,Electronics,177,0.0,NaN,No Sales - 30 Days,33217.59
4769,33,1019,Electronics_Cables_1019,Electronics,205,0.0,NaN,No Sales - 30 Days,31223.55
1497,17,1019,Electronics_Cables_1019,Electronics,204,0.0,NaN,No Sales - 30 Days,31071.24
6565,41,1199,Electronics_Cables_1199,Electronics,206,2.0,3090.0,Severe Overstock,31005.06
1556,17,1199,Electronics_Cables_1199,Electronics,204,1.0,6120.0,Severe Overstock,30704.04
3682,29,1019,Electronics_Cables_1019,Electronics,201,0.0,NaN,No Sales - 30 Days,30614.31
5111,35,1199,Electronics_Cables_1199,Electronics,203,0.0,NaN,No Sales - 30 Days,30553.53


In [92]:
# STEP 13E - Inventory Exposure by Category

category_exposure = (
    inventory_value
    .groupby("category")
    .agg(
        inventory_records=("sku_id", "size"),
        stock_units=("stock_on_hand", "sum"),
        inventory_value=("inventory_cost_value", "sum")
    )
    .reset_index()
)

category_exposure["pct_total_inventory_value"] = (
    category_exposure["inventory_value"] /
    inventory_value["inventory_cost_value"].sum() * 100
).round(2)

category_exposure = category_exposure.sort_values(
    "inventory_value",
    ascending=False
)

print("=" * 70)
print("INVENTORY CAPITAL EXPOSURE BY CATEGORY")
print("=" * 70)

display(category_exposure)

print("\nTOP 3 CATEGORIES BY INVENTORY VALUE")

display(
    category_exposure[
        [
            "category",
            "stock_units",
            "inventory_value",
            "pct_total_inventory_value"
        ]
    ].head(3)
)

INVENTORY CAPITAL EXPOSURE BY CATEGORY


,category,inventory_records,stock_units,inventory_value,pct_total_inventory_value
2,Electronics,1891,213593,17186094.34,44.16
5,Personal Care,949,153981,5089636.24,13.08
3,Grocery,1092,262442,4850998.07,12.47
4,Household,1322,193201,4457724.60,11.46
6,Snacks,1709,331035,3062682.87,7.87
1,Dairy,1088,193173,2769213.13,7.12
0,Beverages,684,144198,1497992.67,3.85



TOP 3 CATEGORIES BY INVENTORY VALUE


,category,stock_units,inventory_value,pct_total_inventory_value
2,Electronics,213593,17186094.34,44.16
5,Personal Care,153981,5089636.24,13.08
3,Grocery,262442,4850998.07,12.47


In [93]:
# STEP 13F - Store-Level Inventory Exposure

store_exposure = (
    inventory_value
    .groupby("store_id")
    .agg(
        inventory_records=("sku_id", "size"),
        stock_units=("stock_on_hand", "sum"),
        inventory_value=("inventory_cost_value", "sum")
    )
    .reset_index()
)

# At-risk inventory by store
store_risk = (
    trapped_inventory
    .groupby("store_id")
    .agg(
        at_risk_units=("stock_on_hand", "sum"),
        at_risk_value=("inventory_cost_value", "sum")
    )
    .reset_index()
)

store_exposure = store_exposure.merge(
    store_risk,
    on="store_id",
    how="left"
)

store_exposure[
    ["at_risk_units", "at_risk_value"]
] = store_exposure[
    ["at_risk_units", "at_risk_value"]
].fillna(0)

store_exposure["pct_value_at_risk"] = (
    store_exposure["at_risk_value"] /
    store_exposure["inventory_value"] * 100
).round(2)

store_exposure = store_exposure.sort_values(
    "at_risk_value",
    ascending=False
)

print("=" * 70)
print("STORE-LEVEL INVENTORY EXPOSURE")
print("=" * 70)

print(f"Stores analysed : {len(store_exposure):,}")

print("\nTOP 10 STORES BY AT-RISK INVENTORY VALUE")

display(
    store_exposure[
        [
            "store_id",
            "stock_units",
            "inventory_value",
            "at_risk_units",
            "at_risk_value",
            "pct_value_at_risk"
        ]
    ].head(10)
)

top10_value = store_exposure.head(10)["at_risk_value"].sum()
total_risk = store_exposure["at_risk_value"].sum()

print(
    f"\nTop 10 stores at-risk value : ${top10_value:,.2f}"
)

print(
    f"Share of total exposure     : "
    f"{top10_value / total_risk * 100:.2f}%"
)

print(
    f"Reconciliation              : "
    f"${total_risk:,.2f}"
)

STORE-LEVEL INVENTORY EXPOSURE
Stores analysed : 50

TOP 10 STORES BY AT-RISK INVENTORY VALUE


,store_id,stock_units,inventory_value,at_risk_units,at_risk_value,pct_value_at_risk
11,12,40717,1095097.26,40717,1095097.26,100.0
28,29,39134,1035721.59,39134,1035721.59,100.0
16,17,38652,1032519.10,38652,1032519.10,100.0
10,11,40250,1031829.90,40250,1031829.90,100.0
34,35,39097,1020900.31,39097,1020900.31,100.0
18,19,40688,1012802.16,40688,1012802.16,100.0
21,22,37015,1006573.75,37015,1006573.75,100.0
7,8,38972,1005342.59,38972,1005342.59,100.0
29,30,36509,1000741.48,36509,1000741.48,100.0
14,15,39548,989396.79,39548,989396.79,100.0



Top 10 stores at-risk value : $10,230,924.93
Share of total exposure     : 26.30%
Reconciliation              : $38,903,748.56


In [94]:
# STEP 14A - Promotion Performance Audit

print("=" * 70)
print("PROMOTION PERFORMANCE AUDIT")
print("=" * 70)

print(f"Promotion records : {len(promotions):,}")
print(f"Unique promotions : {promotions['promo_id'].nunique():,}")

print(
    f"Promotion period  : "
    f"{promotions['start_date'].min()} to "
    f"{promotions['end_date'].max()}"
)

print("\nPROMOTION TYPES")
print(
    promotions["promo_type"]
    .value_counts(dropna=False)
)

print("\nDISCOUNT LEVELS")
print(
    promotions["discount_pct"]
    .value_counts()
    .sort_index()
)

print("\nDATA QUALITY")

print(
    f"Missing promo IDs       : "
    f"{promotions['promo_id'].isna().sum():,}"
)

print(
    f"Missing discount        : "
    f"{promotions['discount_pct'].isna().sum():,}"
)

print(
    f"Invalid date ranges     : "
    f"{(pd.to_datetime(promotions['end_date']) <
        pd.to_datetime(promotions['start_date'])).sum():,}"
)

display(promotions.head(10))

PROMOTION PERFORMANCE AUDIT
Promotion records : 33
Unique promotions : 33
Promotion period  : 2021-01-01 to 2025-08-31

PROMOTION TYPES
promo_type
Eid             10
Ramadan          5
DSF              5
Summer Sale      5
Black Friday     4
National Day     4
Name: count, dtype: int64

DISCOUNT LEVELS
discount_pct
10    5
15    7
20    6
25    8
30    4
35    3
Name: count, dtype: int64

DATA QUALITY
Missing promo IDs       : 0
Missing discount        : 0
Invalid date ranges     : 0


,promo_name,start_date,end_date,discount_pct,promo_type,promo_id
0,Ramadan Sale 2021,2021-04-13,2021-05-12,25,Ramadan,1
1,Eid Al Fitr Sale 2021,2021-05-13,2021-05-15,20,Eid,2
2,Eid Al Adha Sale 2021,2021-07-20,2021-07-22,25,Eid,3
3,Dubai Shopping Festival 2021,2021-01-01,2021-02-28,15,DSF,4
4,Black Friday 2021,2021-11-26,2021-11-28,30,Black Friday,5
5,UAE National Day 2021,2021-12-01,2021-12-03,20,National Day,6
6,Summer Sale 2021,2021-07-01,2021-08-31,10,Summer Sale,7
7,Ramadan Sale 2022,2022-04-02,2022-05-01,25,Ramadan,8
8,Eid Al Fitr Sale 2022,2022-05-02,2022-05-04,25,Eid,9
9,Eid Al Adha Sale 2022,2022-07-10,2022-07-12,25,Eid,10


In [95]:
# STEP 14B - Promotion Calendar Overlap Audit

promotions["start_date"] = pd.to_datetime(promotions["start_date"])
promotions["end_date"] = pd.to_datetime(promotions["end_date"])

overlaps = []

for i in range(len(promotions)):
    for j in range(i + 1, len(promotions)):

        p1 = promotions.iloc[i]
        p2 = promotions.iloc[j]

        overlap_start = max(p1["start_date"], p2["start_date"])
        overlap_end = min(p1["end_date"], p2["end_date"])

        if overlap_start <= overlap_end:
            overlaps.append({
                "promo_1": p1["promo_name"],
                "promo_2": p2["promo_name"],
                "overlap_start": overlap_start,
                "overlap_end": overlap_end,
                "discount_1": p1["discount_pct"],
                "discount_2": p2["discount_pct"]
            })

overlap_df = pd.DataFrame(overlaps)

print("=" * 70)
print("PROMOTION CALENDAR OVERLAP AUDIT")
print("=" * 70)

print(f"Promotions checked : {len(promotions):,}")
print(f"Overlapping pairs  : {len(overlap_df):,}")

if len(overlap_df) == 0:
    print("\nPASS - No promotion date overlaps detected.")
else:
    print("\nWARNING - Promotion overlaps detected.")
    display(overlap_df)

PROMOTION CALENDAR OVERLAP AUDIT
Promotions checked : 33
Overlapping pairs  : 4

WARNING - Promotion overlaps detected.


,promo_1,promo_2,overlap_start,overlap_end,discount_1,discount_2
0,Eid Al Adha Sale 2021,Summer Sale 2021,2021-07-20,2021-07-22,25,10
1,Eid Al Adha Sale 2022,Summer Sale 2022,2022-07-10,2022-07-12,25,10
2,Eid Al Adha Sale 2023,Summer Sale 2023,2023-07-01,2023-07-01,20,20
3,Black Friday 2024,UAE National Day 2024,2024-12-01,2024-12-01,35,15


In [96]:
# STEP 14C - Safe Promotion-to-Sales Mapping
# Match using BOTH transaction date and discount percentage.
# Preserve the master sales dataset.

sales_promo = sales.copy()
sales_promo["date"] = pd.to_datetime(sales_promo["date"])

# Build promotion calendar at daily level
promo_calendar = promotions.copy()

promo_calendar["promo_date"] = promo_calendar.apply(
    lambda r: pd.date_range(r["start_date"], r["end_date"]),
    axis=1
)

promo_calendar = (
    promo_calendar
    .explode("promo_date")
    .reset_index(drop=True)
)

# Keep only required fields
promo_calendar = promo_calendar[
    [
        "promo_id",
        "promo_name",
        "promo_type",
        "discount_pct",
        "promo_date"
    ]
]

# Count how many promotions could match each date + discount
match_counts = (
    promo_calendar
    .groupby(["promo_date", "discount_pct"])
    .size()
    .reset_index(name="possible_promos")
)

# Keep unique matches only
unique_calendar = (
    promo_calendar
    .merge(
        match_counts,
        on=["promo_date", "discount_pct"],
        how="left"
    )
)

unique_calendar.loc[
    unique_calendar["possible_promos"] > 1,
    ["promo_id", "promo_name", "promo_type"]
] = [pd.NA, "AMBIGUOUS", "AMBIGUOUS"]

# One row per date + discount
unique_calendar = (
    unique_calendar
    .drop_duplicates(["promo_date", "discount_pct"])
)

# Safe left merge
sales_promo = sales_promo.merge(
    unique_calendar,
    left_on=["date", "discount_pct"],
    right_on=["promo_date", "discount_pct"],
    how="left"
)

sales_promo["promotion_status"] = "No Promotion Match"

sales_promo.loc[
    sales_promo["promo_name"].notna(),
    "promotion_status"
] = "Matched"

sales_promo.loc[
    sales_promo["promo_name"].eq("AMBIGUOUS"),
    "promotion_status"
] = "Ambiguous"

print("=" * 70)
print("SAFE PROMOTION-TO-SALES MAPPING")
print("=" * 70)

print(f"Input sales rows       : {len(sales):,}")
print(f"Output sales rows      : {len(sales_promo):,}")
print(f"Row difference         : {len(sales_promo) - len(sales):,}")

print("\nPROMOTION MATCH STATUS")
print(sales_promo["promotion_status"].value_counts(dropna=False))

print("\nPROMOTION REVENUE")
promo_summary = (
    sales_promo[
        sales_promo["promotion_status"] == "Matched"
    ]
    .groupby(["promo_name", "promo_type"], as_index=False)
    .agg(
        transactions=("sku_id", "size"),
        units=("quantity", "sum"),
        revenue=("total_value", "sum")
    )
    .sort_values("revenue", ascending=False)
)

display(promo_summary.head(15))

if len(sales_promo) == len(sales):
    print("\nPASS - Sales row count preserved after promotion mapping.")
else:
    print("\nWARNING - Sales row count changed.")

SAFE PROMOTION-TO-SALES MAPPING
Input sales rows       : 641,843
Output sales rows      : 641,843
Row difference         : 0

PROMOTION MATCH STATUS
promotion_status
No Promotion Match    437475
Matched               204181
Ambiguous                187
Name: count, dtype: int64

PROMOTION REVENUE


,promo_name,promo_type,transactions,units,revenue
7,Dubai Shopping Festival 2024,DSF,17343,54796,2191662.35
4,Dubai Shopping Festival 2021,DSF,17729,55610,2180403.54
8,Dubai Shopping Festival 2025,DSF,17070,53662,2108445.48
6,Dubai Shopping Festival 2023,DSF,16429,51785,2047147.65
5,Dubai Shopping Festival 2022,DSF,17111,54200,1932467.79
27,Summer Sale 2024,Summer Sale,13720,42925,1808236.02
25,Summer Sale 2022,Summer Sale,13397,42188,1797225.90
24,Summer Sale 2021,Summer Sale,13199,41675,1762909.15
28,Summer Sale 2025,Summer Sale,13237,41560,1751577.87
26,Summer Sale 2023,Summer Sale,12984,41094,1509721.97



PASS - Sales row count preserved after promotion mapping.


In [97]:
# STEP 14D - Promotion Profitability Analysis

# Add profitability data to safely mapped promotion dataset
promo_profit = sales_promo.merge(
    skus[
        ["sku_id", "cost_price"]
    ],
    on="sku_id",
    how="left"
)

promo_profit["cogs"] = (
    promo_profit["quantity"] *
    promo_profit["cost_price"]
)

promo_profit["gross_profit"] = (
    promo_profit["total_value"] -
    promo_profit["cogs"]
)

print("=" * 70)
print("PROMOTION PROFITABILITY ANALYSIS")
print("=" * 70)

print(f"Input rows              : {len(sales_promo):,}")
print(f"Output rows             : {len(promo_profit):,}")
print(f"Missing cost prices     : {promo_profit['cost_price'].isna().sum():,}")

# Analyse only confidently matched promotions
matched_promo = promo_profit[
    promo_profit["promotion_status"] == "Matched"
].copy()

promotion_performance = (
    matched_promo
    .groupby(
        ["promo_name", "promo_type", "discount_pct"],
        as_index=False
    )
    .agg(
        transactions=("sku_id", "size"),
        units=("quantity", "sum"),
        revenue=("total_value", "sum"),
        gross_profit=("gross_profit", "sum")
    )
)

promotion_performance["gross_margin_pct"] = (
    promotion_performance["gross_profit"] /
    promotion_performance["revenue"] * 100
)

promotion_performance = promotion_performance.sort_values(
    "gross_profit",
    ascending=False
)

print(f"\nMatched transactions    : {len(matched_promo):,}")
print(f"Matched promotion sales : ${matched_promo['total_value'].sum():,.2f}")
print(f"Matched gross profit    : ${matched_promo['gross_profit'].sum():,.2f}")

print("\nTOP 15 PROMOTIONS BY GROSS PROFIT")

display(
    promotion_performance[
        [
            "promo_name",
            "promo_type",
            "discount_pct",
            "transactions",
            "units",
            "revenue",
            "gross_profit",
            "gross_margin_pct"
        ]
    ].head(15)
)

print("\nBOTTOM 10 PROMOTIONS BY GROSS MARGIN %")

display(
    promotion_performance.sort_values(
        "gross_margin_pct"
    ).head(10)
)

if len(promo_profit) == len(sales_promo):
    print("\nPASS - Row count preserved after profitability analysis.")
else:
    print("\nWARNING - Row count changed.")

PROMOTION PROFITABILITY ANALYSIS
Input rows              : 641,843
Output rows             : 641,843
Missing cost prices     : 0

Matched transactions    : 204,181
Matched promotion sales : $24,885,074.04
Matched gross profit    : $4,936,371.85

TOP 15 PROMOTIONS BY GROSS PROFIT


,promo_name,promo_type,discount_pct,transactions,units,revenue,gross_profit,gross_margin_pct
7,Dubai Shopping Festival 2024,DSF,15.0,17343,54796,2191662.35,477896.30,21.805197
27,Summer Sale 2024,Summer Sale,10.0,13720,42925,1808236.02,472677.94,26.140279
4,Dubai Shopping Festival 2021,DSF,15.0,17729,55610,2180403.54,471074.91,21.604942
25,Summer Sale 2022,Summer Sale,10.0,13397,42188,1797225.90,468776.18,26.083320
24,Summer Sale 2021,Summer Sale,10.0,13199,41675,1762909.15,460960.47,26.147716
28,Summer Sale 2025,Summer Sale,10.0,13237,41560,1751577.87,456067.38,26.037517
8,Dubai Shopping Festival 2025,DSF,15.0,17070,53662,2108445.48,454558.91,21.558960
6,Dubai Shopping Festival 2023,DSF,15.0,16429,51785,2047147.65,442283.21,21.604852
26,Summer Sale 2023,Summer Sale,20.0,12984,41094,1509721.97,253714.89,16.805405
5,Dubai Shopping Festival 2022,DSF,25.0,17111,54200,1932467.79,219516.03,11.359363



BOTTOM 10 PROMOTIONS BY GROSS MARGIN %


,promo_name,promo_type,discount_pct,transactions,units,revenue,gross_profit,gross_margin_pct
3,Black Friday 2024,Black Friday,35.0,551,1707,48490.05,-1580.94,-3.260339
1,Black Friday 2022,Black Friday,35.0,732,2269,63947.37,-1483.17,-2.319360
2,Black Friday 2023,Black Friday,35.0,829,2576,79107.22,-1294.51,-1.636399
12,Eid Al Adha Sale 2024,Eid,30.0,825,2640,84479.30,3987.49,4.720079
0,Black Friday 2021,Black Friday,30.0,875,2709,90972.75,4334.70,4.764833
18,Eid Al Fitr Sale 2025,Eid,30.0,682,2140,71550.84,3414.20,4.771712
23,Ramadan Sale 2025,Ramadan,30.0,7818,24393,781854.52,38054.30,4.867184
20,Ramadan Sale 2022,Ramadan,25.0,7691,24036,838062.12,93993.19,11.215540
19,Ramadan Sale 2021,Ramadan,25.0,8654,27242,953281.38,107324.88,11.258468
13,Eid Al Adha Sale 2025,Eid,25.0,1172,3698,127903.01,14490.29,11.329124



PASS - Row count preserved after profitability analysis.


In [98]:
# STEP 14E - Discount Level vs Profitability

discount_performance = (
    matched_promo
    .groupby("discount_pct", as_index=False)
    .agg(
        transactions=("sku_id", "size"),
        units=("quantity", "sum"),
        revenue=("total_value", "sum"),
        gross_profit=("gross_profit", "sum")
    )
)

discount_performance["gross_margin_pct"] = (
    discount_performance["gross_profit"] /
    discount_performance["revenue"] * 100
)

discount_performance["revenue_per_transaction"] = (
    discount_performance["revenue"] /
    discount_performance["transactions"]
)

discount_performance = discount_performance.sort_values("discount_pct")

print("=" * 70)
print("DISCOUNT LEVEL vs PROFITABILITY")
print("=" * 70)

display(discount_performance)

# Loss-making transactions by discount level
loss_sales = matched_promo[
    matched_promo["gross_profit"] < 0
]

loss_by_discount = (
    loss_sales
    .groupby("discount_pct", as_index=False)
    .agg(
        loss_transactions=("sku_id", "size"),
        revenue_at_risk=("total_value", "sum"),
        gross_loss=("gross_profit", "sum")
    )
)

loss_by_discount["gross_loss"] = (
    loss_by_discount["gross_loss"].abs()
)

print("\nLOSS-MAKING PROMOTIONAL SALES")
display(loss_by_discount)

print("\nCHECK")
print(f"Matched rows       : {len(matched_promo):,}")
print(f"Loss-making rows   : {len(loss_sales):,}")
print(
    f"Loss-making rate   : "
    f"{len(loss_sales) / len(matched_promo) * 100:.2f}%"
)
print(
    f"Gross loss         : "
    f"${abs(loss_sales['gross_profit'].sum()):,.2f}"
)

DISCOUNT LEVEL vs PROFITABILITY


,discount_pct,transactions,units,revenue,gross_profit,gross_margin_pct,revenue_per_transaction
0,10.0,54566,171448,7246784.30,1891044.16,26.094942,132.807688
1,15.0,77212,243193,9610260.90,2079759.61,21.641032,124.465898
2,20.0,23312,73849,2716880.21,456625.27,16.806971,116.544278
3,25.0,36779,115981,4090746.58,463510.74,11.330713,111.225063
4,30.0,10200,31882,1028857.41,49790.69,4.839416,100.868374
5,35.0,2112,6552,191544.64,-4358.62,-2.275511,90.693485



LOSS-MAKING PROMOTIONAL SALES


,discount_pct,loss_transactions,revenue_at_risk,gross_loss
0,25.0,5680,627126.40,22820.71
1,30.0,3413,329664.48,21527.71
2,35.0,1198,110182.87,10670.63



CHECK
Matched rows       : 204,181
Loss-making rows   : 10,291
Loss-making rate   : 5.04%
Gross loss         : $55,019.05


In [99]:
# STEP 14F - Promotion Guardrail Compliance

promo_guardrail = matched_promo.merge(
    sku_guardrail[
        ["sku_id", "max_safe_discount_pct"]
    ],
    on="sku_id",
    how="left"
)

promo_guardrail["guardrail_breach"] = (
    promo_guardrail["discount_pct"] >
    promo_guardrail["max_safe_discount_pct"]
)

promo_guardrail["discount_excess_pct"] = (
    promo_guardrail["discount_pct"] -
    promo_guardrail["max_safe_discount_pct"]
).clip(lower=0)

print("=" * 70)
print("PROMOTION GUARDRAIL COMPLIANCE")
print("=" * 70)

print(f"Promotional rows analysed : {len(promo_guardrail):,}")
print(
    f"Guardrail breaches        : "
    f"{promo_guardrail['guardrail_breach'].sum():,}"
)
print(
    f"Breach rate               : "
    f"{promo_guardrail['guardrail_breach'].mean()*100:.2f}%"
)

# Breaches by discount level
breach_summary = (
    promo_guardrail
    .groupby("discount_pct", as_index=False)
    .agg(
        promotional_rows=("sku_id", "size"),
        breaches=("guardrail_breach", "sum"),
        revenue=("total_value", "sum"),
        gross_profit=("gross_profit", "sum")
    )
)

breach_summary["breach_rate_pct"] = (
    breach_summary["breaches"] /
    breach_summary["promotional_rows"] * 100
)

print("\nGUARDRAIL BREACH BY DISCOUNT LEVEL")
display(breach_summary)

# Most frequently breached SKUs
top_breach_skus = (
    promo_guardrail[
        promo_guardrail["guardrail_breach"]
    ]
    .groupby("sku_id", as_index=False)
    .agg(
        breach_transactions=("sku_id", "size"),
        revenue_exposed=("total_value", "sum"),
        gross_profit=("gross_profit", "sum"),
        max_safe_discount_pct=("max_safe_discount_pct", "first")
    )
    .sort_values("breach_transactions", ascending=False)
)

top_breach_skus = top_breach_skus.merge(
    skus[["sku_id", "sku_name", "category"]],
    on="sku_id",
    how="left"
)

print("\nTOP 10 SKUs BREACHING DISCOUNT GUARDRAIL")

display(
    top_breach_skus[
        [
            "sku_id",
            "sku_name",
            "category",
            "max_safe_discount_pct",
            "breach_transactions",
            "revenue_exposed",
            "gross_profit"
        ]
    ].head(10)
)

if len(promo_guardrail) == len(matched_promo):
    print("\nPASS - Promotional row count preserved.")
else:
    print("\nWARNING - Row count changed.")

PROMOTION GUARDRAIL COMPLIANCE
Promotional rows analysed : 204,181
Guardrail breaches        : 10,314
Breach rate               : 5.05%

GUARDRAIL BREACH BY DISCOUNT LEVEL


,discount_pct,promotional_rows,breaches,revenue,gross_profit,breach_rate_pct
0,10.0,54566,0,7246784.30,1891044.16,0.000000
1,15.0,77212,0,9610260.90,2079759.61,0.000000
2,20.0,23312,0,2716880.21,456625.27,0.000000
3,25.0,36779,5680,4090746.58,463510.74,15.443596
4,30.0,10200,3452,1028857.41,49790.69,33.843137
5,35.0,2112,1182,191544.64,-4358.62,55.965909



TOP 10 SKUs BREACHING DISCOUNT GUARDRAIL


,sku_id,sku_name,category,max_safe_discount_pct,breach_transactions,revenue_exposed,gross_profit
0,1121,Grocery_Rice_1121,Grocery,20.36,287,22436.09,-1785.97
1,1165,Personal Care_Skincare_1165,Personal Care,23.99,276,11162.48,-370.07
2,1068,Electronics_Chargers_1068,Electronics,21.73,270,58272.11,-3802.51
3,1156,Personal Care_Soap_1156,Personal Care,24.48,270,21070.39,-569.77
4,1139,Dairy_Cream_1139,Dairy,21.71,269,14204.92,-872.52
5,1030,Dairy_Cream_1030,Dairy,22.37,269,13911.01,-752.26
6,1096,Electronics_Mobile Accessories_1096,Electronics,21.68,263,69861.19,-4849.47
7,1072,Electronics_Chargers_1072,Electronics,21.23,263,141219.60,-9667.08
8,1194,Dairy_Butter_1194,Dairy,22.53,260,5776.88,-340.32
9,1160,Snacks_Nuts_1160,Snacks,22.33,255,3664.20,-204.60



PASS - Promotional row count preserved.


In [100]:
# STEP 15A - Prepare Power BI Sales Dataset

powerbi_sales = sales_profit.copy()

print("=" * 65)
print("POWER BI SALES DATASET CHECK")
print("=" * 65)

print(f"Rows           : {len(powerbi_sales):,}")
print(f"Columns        : {len(powerbi_sales.columns):,}")
print(f"Unique stores  : {powerbi_sales['store_id'].nunique():,}")
print(f"Unique SKUs    : {powerbi_sales['sku_id'].nunique():,}")
print(f"Revenue        : ${powerbi_sales['total_value'].sum():,.2f}")
print(f"Gross Profit   : ${powerbi_sales['gross_profit'].sum():,.2f}")

if len(powerbi_sales) == len(sales):
    print("\nPASS - Power BI sales row count reconciled.")
else:
    print("\nWARNING - Power BI sales row count mismatch.")

POWER BI SALES DATASET CHECK
Rows           : 641,843
Columns        : 15
Unique stores  : 50
Unique SKUs    : 200
Revenue        : $73,214,931.30
Gross Profit   : $21,098,695.72

PASS - Power BI sales row count reconciled.


In [101]:
# STEP 15B - Export Power BI Sales Dataset

powerbi_sales.to_csv(
    "powerbi_sales.csv",
    index=False
)

print("=" * 65)
print("POWER BI SALES EXPORT")
print("=" * 65)

print("File created : powerbi_sales.csv")
print(f"Rows exported: {len(powerbi_sales):,}")
print(f"Columns      : {len(powerbi_sales.columns):,}")

print("\nPASS - Sales dataset exported successfully.")

POWER BI SALES EXPORT
File created : powerbi_sales.csv
Rows exported: 641,843
Columns      : 15

PASS - Sales dataset exported successfully.


In [102]:
# STEP 15C - Export Power BI Inventory Dataset

powerbi_inventory = inventory_value.copy()

powerbi_inventory.to_csv(
    "powerbi_inventory.csv",
    index=False
)

print("=" * 65)
print("POWER BI INVENTORY EXPORT")
print("=" * 65)

print("File created : powerbi_inventory.csv")
print(f"Rows exported: {len(powerbi_inventory):,}")
print(f"Columns      : {len(powerbi_inventory.columns):,}")

print(
    f"Inventory value: "
    f"${powerbi_inventory['inventory_cost_value'].sum():,.2f}"
)

if len(powerbi_inventory) == len(inventory):
    print("\nPASS - Inventory dataset exported successfully.")
else:
    print("\nWARNING - Inventory row count mismatch.")

POWER BI INVENTORY EXPORT
File created : powerbi_inventory.csv
Rows exported: 8,735
Columns      : 16
Inventory value: $38,914,341.92

PASS - Inventory dataset exported successfully.


In [103]:
# STEP 15D - Export Power BI Promotion Dataset

powerbi_promotions = promo_guardrail.copy()

powerbi_promotions.to_csv(
    "powerbi_promotions.csv",
    index=False
)

print("=" * 65)
print("POWER BI PROMOTION EXPORT")
print("=" * 65)

print("File created : powerbi_promotions.csv")
print(f"Rows exported: {len(powerbi_promotions):,}")
print(f"Columns      : {len(powerbi_promotions.columns):,}")

print(
    f"Matched promotion revenue: "
    f"${powerbi_promotions['total_value'].sum():,.2f}"
)

print(
    f"Guardrail breaches       : "
    f"{powerbi_promotions['guardrail_breach'].sum():,}"
)

print("\nPASS - Promotion dataset exported successfully.")

POWER BI PROMOTION EXPORT
File created : powerbi_promotions.csv
Rows exported: 204,181
Columns      : 21
Matched promotion revenue: $24,885,074.04
Guardrail breaches       : 10,314

PASS - Promotion dataset exported successfully.


In [104]:
# STEP 15E - Export Power BI Master Tables

stores.to_csv("powerbi_stores.csv", index=False)
skus.to_csv("powerbi_skus.csv", index=False)
customers.to_csv("powerbi_customers.csv", index=False)

print("=" * 65)
print("POWER BI MASTER TABLES EXPORT")
print("=" * 65)

print(f"Stores    : {len(stores):,} rows -> powerbi_stores.csv")
print(f"SKUs      : {len(skus):,} rows -> powerbi_skus.csv")
print(f"Customers : {len(customers):,} rows -> powerbi_customers.csv")

print("\nPASS - Master tables exported successfully.")

POWER BI MASTER TABLES EXPORT
Stores    : 50 rows -> powerbi_stores.csv
SKUs      : 200 rows -> powerbi_skus.csv
Customers : 5,000 rows -> powerbi_customers.csv

PASS - Master tables exported successfully.


In [105]:
# STEP 15F - Final Power BI Export Validation

import os

export_files = [
    "powerbi_sales.csv",
    "powerbi_inventory.csv",
    "powerbi_promotions.csv",
    "powerbi_stores.csv",
    "powerbi_skus.csv",
    "powerbi_customers.csv"
]

print("=" * 70)
print("FINAL POWER BI EXPORT VALIDATION")
print("=" * 70)

all_ok = True

for file in export_files:
    if os.path.exists(file):
        size_mb = os.path.getsize(file) / (1024 * 1024)
        print(f"PASS  {file:<28} {size_mb:>8.2f} MB")
    else:
        print(f"FAIL  {file:<28} FILE MISSING")
        all_ok = False

print("=" * 70)

if all_ok:
    print("PASS - ALL POWER BI DATASETS READY.")
else:
    print("WARNING - One or more files are missing.")

FINAL POWER BI EXPORT VALIDATION
PASS  powerbi_sales.csv               79.28 MB
PASS  powerbi_inventory.csv            1.10 MB
PASS  powerbi_promotions.csv          30.09 MB
PASS  powerbi_stores.csv               0.00 MB
PASS  powerbi_skus.csv                 0.01 MB
PASS  powerbi_customers.csv            0.22 MB
PASS - ALL POWER BI DATASETS READY.


# Executive Summary

This project analyses 641,843 retail sales records across 50 stores and 200 SKUs to identify operational and profitability risks.

## Key Findings

- Revenue analysed: $73.21M  

- Gross profit: $21.10M

- Overall gross margin: 28.82%
- 10,291 loss-making sales records were identified.
- SKU-level discount guardrails identified approximately $55K in historical gross-margin leakage.
- Promotion risk increased sharply above 25% discount.
- 35% discount campaigns produced negative aggregate gross margin.
- Inventory analysis identified significant working-capital exposure using 30-day sales velocity and Days of Supply.
- Electronics represented 44.16% of total inventory value.

## Business Recommendations

1. Replace blanket discounts with SKU-level discount guardrails.
2. Review promotions above 25% before approval.
3. Prioritize slow-moving and no-sale inventory for rebalancing or markdown review.
4. Focus working-capital actions on high-value categories, especially Electronics.
5. Use sales velocity and Days of Supply alongside reorder points for inventory planning.